In [8]:
import sqlite3
import pandas as pd

# =====================================================================
# STAP 1 & 2: CONNECTIES EN MAPPING
# =====================================================================
conn_acc_verkoop = sqlite3.connect("C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_1_Accessoireverkoop.db")
conn_fiets_verkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_2_Fietsverkoop.db')
conn_onderhoud = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_3_Onderhoud.db')
conn_acc_inkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_4_Accessoire_Inkoop.db')
conn_fiets_inkoop = sqlite3.connect('C:\\Users\\xande\\School\\DEAI - WC Source Data Model Implementation\\BikeToDrive_5_Fiets_Inkoop.db')

db_mapping = {
    conn_acc_inkoop: ['Leverancier', 'Accessoire', 'Accessoire_Inkoop'],
    conn_fiets_inkoop: ['Fabrikant', 'Fiets', 'Fiets_Inkoop'],
    conn_onderhoud: ['Onderhoud', 'Monteur', 'Filiaal', 'Fiets', 'Fabrikant'],
    conn_acc_verkoop: ['Accessoire', 'Accessoire_Verkoop', 'Klant', 'Leverancier', 'Filiaal', 'Monteur'],
    conn_fiets_verkoop: ['Fiets', 'Fiets_Verkoop', 'Klant', 'Fabrikant', 'Filiaal', 'Monteur']
}

# =====================================================================
# STAP 3: DATAFRAMES MAKEN (Zonder TRIAL kolommen, met samenvoeging)
# =====================================================================
dataframes = {}
for conn, tables in db_mapping.items():
    for table in tables:
        query = f"SELECT * FROM {table}"
        df = pd.read_sql_query(query, conn)
        
        # Gooi de onnodige 'TRIAL' kolommen weg
        df = df.loc[:, ~df.columns.str.startswith('TRIAL')]
        
        # Samenvoegen (Union) als de stamtabel al in een andere DB voorkwam
        if table in dataframes:
            dataframes[table] = pd.concat([dataframes[table], df]).drop_duplicates()
        else:
            dataframes[table] = df

# =====================================================================
# STAP 4: ASSOCIATIES MAKEN (Met Suffixes en correcte ID-sleutels)
# =====================================================================
linked_dataframes = {}

# Basis tabellen direct overnemen
for basis_tabel in ['Fiets', 'Accessoire', 'Klant', 'Leverancier', 'Fabrikant', 'Filiaal', 'Monteur']:
    linked_dataframes[basis_tabel] = dataframes[basis_tabel]

# --- FIETS VERKOOP KOPPELEN ---
linked_dataframes['Fiets_Verkoop'] = dataframes['Fiets_Verkoop'].merge(
    dataframes['Fiets'], left_on='fiets', right_on='fietsnr', how='left')
linked_dataframes['Fiets_Verkoop'] = linked_dataframes['Fiets_Verkoop'].merge(
    dataframes['Klant'], left_on='klant', right_on='klantnr', how='left', suffixes=('', '_klant'))
linked_dataframes['Fiets_Verkoop'] = linked_dataframes['Fiets_Verkoop'].merge(
    dataframes['Fabrikant'], left_on='fabrikant', right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))
linked_dataframes['Fiets_Verkoop'] = linked_dataframes['Fiets_Verkoop'].merge(
    dataframes['Monteur'], left_on='monteur', right_on='monteurnr', how='left', suffixes=('', '_monteur'))
linked_dataframes['Fiets_Verkoop'] = linked_dataframes['Fiets_Verkoop'].merge(
    dataframes['Filiaal'], left_on='filiaal', right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))

# --- FIETS INKOOP KOPPELEN ---
linked_dataframes['Fiets_Inkoop'] = dataframes['Fiets_Inkoop'].merge(
    dataframes['Fiets'], left_on='fiets', right_on='fietsnr', how='left')
linked_dataframes['Fiets_Inkoop'] = linked_dataframes['Fiets_Inkoop'].merge(
    dataframes['Fabrikant'], left_on='fabrikant', right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))

# --- ACCESSOIRE VERKOOP KOPPELEN ---
linked_dataframes['Accessoire_Verkoop'] = dataframes['Accessoire_Verkoop'].merge(
    dataframes['Accessoire'], left_on='accessoire', right_on='accessoirenr', how='left')
linked_dataframes['Accessoire_Verkoop'] = linked_dataframes['Accessoire_Verkoop'].merge(
    dataframes['Klant'], left_on='klant', right_on='klantnr', how='left', suffixes=('', '_klant'))
linked_dataframes['Accessoire_Verkoop'] = linked_dataframes['Accessoire_Verkoop'].merge(
    dataframes['Leverancier'], left_on='leverancier', right_on='leveranciernr', how='left', suffixes=('', '_leverancier'))
linked_dataframes['Accessoire_Verkoop'] = linked_dataframes['Accessoire_Verkoop'].merge(
    dataframes['Monteur'], left_on='monteur', right_on='monteurnr', how='left', suffixes=('', '_monteur'))
linked_dataframes['Accessoire_Verkoop'] = linked_dataframes['Accessoire_Verkoop'].merge(
    dataframes['Filiaal'], left_on='filiaal', right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))

# --- ACCESSOIRE INKOOP KOPPELEN ---
linked_dataframes['Accessoire_Inkoop'] = dataframes['Accessoire_Inkoop'].merge(
    dataframes['Accessoire'], left_on='accessoire', right_on='accessoirenr', how='left')
linked_dataframes['Accessoire_Inkoop'] = linked_dataframes['Accessoire_Inkoop'].merge(
    dataframes['Leverancier'], left_on='leverancier', right_on='leveranciernr', how='left', suffixes=('', '_leverancier'))

# --- ONDERHOUD KOPPELEN ---
linked_dataframes['Onderhoud'] = dataframes['Onderhoud'].merge(
    dataframes['Fiets'], left_on='fiets', right_on='fietsnr', how='left')
linked_dataframes['Onderhoud'] = linked_dataframes['Onderhoud'].merge(
    dataframes['Fabrikant'], left_on='fabrikant', right_on='fabrikantnr', how='left', suffixes=('', '_fabrikant'))
linked_dataframes['Onderhoud'] = linked_dataframes['Onderhoud'].merge(
    dataframes['Monteur'], left_on='monteur', right_on='monteurnr', how='left', suffixes=('', '_monteur'))
linked_dataframes['Onderhoud'] = linked_dataframes['Onderhoud'].merge(
    dataframes['Filiaal'], left_on='filiaal', right_on='filiaalnr', how='left', suffixes=('', '_filiaal'))


# =====================================================================
# STAP 5: BASIS MULTIPLICITEITEN (Binnen het gecentraliseerde model)
# =====================================================================
multiplicities = {}

multiplicities['Stamgegevens (Totaal Uniek)'] = {
    'Fietsen': linked_dataframes['Fiets']['fietsnr'].nunique(),
    'Accessoires': linked_dataframes['Accessoire']['accessoirenr'].nunique(),
    'Klanten': linked_dataframes['Klant']['klantnr'].nunique(),
    'Leveranciers': linked_dataframes['Leverancier']['leveranciernr'].nunique(),
    'Fabrikanten': linked_dataframes['Fabrikant']['fabrikantnr'].nunique(),
    'Filialen': linked_dataframes['Filiaal']['filiaalnr'].nunique(),
    'Monteurs': linked_dataframes['Monteur']['monteurnr'].nunique()
}

multiplicities['Transacties (Aantal uniek gebruikte entiteiten)'] = { 
    'Fietsen verkocht': linked_dataframes['Fiets_Verkoop']['fiets'].nunique(),
    'Fietsen ingekocht': linked_dataframes['Fiets_Inkoop']['fiets'].nunique(),
    'Fietsen in onderhoud': linked_dataframes['Onderhoud']['fiets'].nunique(),
    'Accessoires verkocht': linked_dataframes['Accessoire_Verkoop']['accessoire'].nunique(),
    'Accessoires ingekocht': linked_dataframes['Accessoire_Inkoop']['accessoire'].nunique()
}

print("--- BASIS MULTIPLICITEITEN ---")
for categorie, data in multiplicities.items():
    print(f"\n{categorie}:")
    for key, val in data.items():
        print(f"  {key}: {val}")


# =====================================================================
# STAP 6: DATABASE OVERSCHRIJDENDE MULTIPLICITEITEN (Stamtabellen)
# =====================================================================
pk_mapping = {
    'Accessoire': 'accessoirenr', 'Leverancier': 'leveranciernr', 
    'Monteur': 'monteurnr', 'Filiaal': 'filiaalnr', 
    'Fiets': 'fietsnr', 'Fabrikant': 'fabrikantnr', 'Klant': 'klantnr'
}

db_names = {
    conn_acc_verkoop: "Acc_Verkoop", conn_fiets_verkoop: "Fiets_Verkoop",
    conn_onderhoud: "Onderhoud", conn_acc_inkoop: "Acc_Inkoop",
    conn_fiets_inkoop: "Fiets_Inkoop"
}

def bepaal_multipliciteit(set_a, set_b):
    if set_a == set_b:
        return "(1..1) --- (1..1)"
    elif set_a.issubset(set_b):
        return "(1..1) --- (0..1)"
    elif set_b.issubset(set_a):
        return "(0..1) --- (1..1)"
    else:
        return "(0..1) --- (0..1)"

print("\n\n--- MULTIPLICITEITEN DATABASE-OVERSCHRIJDEND (Master Data) ---")
for tabel, pk in pk_mapping.items():
    print(f"\nEntiteit: {tabel}")
    
    sets_per_db = {}
    for conn, tables in db_mapping.items():
        if tabel in tables:
            query = f"SELECT {pk} FROM {tabel}"
            # Ophalen direct uit de specifieke DB voor vergelijking
            df_temp = pd.read_sql_query(query, conn).dropna() 
            db_naam = db_names[conn]
            sets_per_db[db_naam] = set(df_temp[pk])
            
    db_lijst = list(sets_per_db.keys())
    
    if len(db_lijst) <= 1:
        print(f"  Komt slechts in 1 database voor ({db_lijst[0]}).")
        continue

    for i in range(len(db_lijst)):
        for j in range(i + 1, len(db_lijst)):
            db1 = db_lijst[i]
            db2 = db_lijst[j]
            mult = bepaal_multipliciteit(sets_per_db[db1], sets_per_db[db2])
            print(f"  {db1} {mult} {db2}")

--- BASIS MULTIPLICITEITEN ---

Stamgegevens (Totaal Uniek):
  Fietsen: 75
  Accessoires: 13
  Klanten: 25
  Leveranciers: 5
  Fabrikanten: 11
  Filialen: 5
  Monteurs: 15

Transacties (Aantal uniek gebruikte entiteiten):
  Fietsen verkocht: 68
  Fietsen ingekocht: 54
  Fietsen in onderhoud: 24
  Accessoires verkocht: 10
  Accessoires ingekocht: 13


--- MULTIPLICITEITEN DATABASE-OVERSCHRIJDEND (Master Data) ---

Entiteit: Accessoire
  Acc_Inkoop (0..1) --- (1..1) Acc_Verkoop

Entiteit: Leverancier
  Acc_Inkoop (1..1) --- (1..1) Acc_Verkoop

Entiteit: Monteur
  Onderhoud (0..1) --- (1..1) Acc_Verkoop
  Onderhoud (0..1) --- (1..1) Fiets_Verkoop
  Acc_Verkoop (1..1) --- (1..1) Fiets_Verkoop

Entiteit: Filiaal
  Onderhoud (0..1) --- (1..1) Acc_Verkoop
  Onderhoud (0..1) --- (1..1) Fiets_Verkoop
  Acc_Verkoop (1..1) --- (1..1) Fiets_Verkoop

Entiteit: Fiets
  Fiets_Inkoop (0..1) --- (1..1) Onderhoud
  Fiets_Inkoop (1..1) --- (1..1) Fiets_Verkoop
  Onderhoud (1..1) --- (0..1) Fiets_Verkoop
